In [1]:
"""Sanity-check `num_labels` vs label_dict, YAML, src.utils, artifacts, and CSV data."""
from __future__ import annotations

import ast
import json
import sys
from pathlib import Path

import pandas as pd

# Assume notebook cwd is `Project/` (Kernel > Restart and run from Project if imports fail).
PROJECT = Path(".").resolve()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.config import TrainConfig
from src.utils import EMOTION_LABELS, NUM_LABELS

LABEL_JSON = PROJECT / "data" / "vigoemotions" / "label_dict.json"
YAML_PATH = PROJECT / "configs" / "visobert_baseline.yaml"
ARTIFACT_CONFIG = PROJECT / "artifacts" / "visobert-baseline-v1" / "config.json"


def _label_indices_from_csv(csv_path: Path) -> set[int]:
    df = pd.read_csv(csv_path)
    out: set[int] = set()
    for cell in df["labels"].dropna():
        s = str(cell).strip()
        if not s:
            continue
        try:
            v = ast.literal_eval(s)
        except (SyntaxError, ValueError):
            continue
        if isinstance(v, int):
            out.add(v)
        else:
            out.update(int(x) for x in v)
    return out


# --- 1) label_dict.json ---
with LABEL_JSON.open(encoding="utf-8") as f:
    label_dict = json.load(f)
n_json = len(label_dict)
print(f"label_dict.json entries : {n_json}")

# --- 2) src.utils (canonical order) ---
print(f"src.utils NUM_LABELS    : {NUM_LABELS} (EMOTION_LABELS len)")

# --- 3) YAML ---
cfg = TrainConfig.from_yaml(YAML_PATH)
print(f"YAML num_labels         : {cfg.num_labels}")

# --- 4) Last downloaded training snapshot (optional) ---
if ARTIFACT_CONFIG.exists():
    with ARTIFACT_CONFIG.open(encoding="utf-8") as f:
        art = json.load(f)
    print(f"artifacts config.json   : num_labels = {art.get('num_labels')}")
else:
    print(f"artifacts config.json   : (missing) {ARTIFACT_CONFIG}")

# --- 5) Agreement ---
issues = []
if n_json != NUM_LABELS:
    issues.append(f"label_dict.json ({n_json}) != NUM_LABELS ({NUM_LABELS})")
if cfg.num_labels != NUM_LABELS:
    issues.append(f"YAML num_labels ({cfg.num_labels}) != NUM_LABELS ({NUM_LABELS})")
if ARTIFACT_CONFIG.exists() and art.get("num_labels") != NUM_LABELS:
    issues.append(f"artifact num_labels ({art.get('num_labels')}) != NUM_LABELS ({NUM_LABELS})")

# --- 6) label_dict order vs EMOTION_LABELS ---
for i, name in label_dict.items():
    idx = int(i)
    if idx >= len(EMOTION_LABELS) or EMOTION_LABELS[idx] != name:
        issues.append(f"label_dict id {i} -> {name!r} mismatches EMOTION_LABELS[{idx}]")

# --- 7) CSV label index range ---
for split in ("train", "val", "test"):
    p = PROJECT / "data" / "vigoemotions" / f"{split}.csv"
    if not p.exists():
        print(f"{split}.csv: missing ({p})")
        continue
    idx_set = _label_indices_from_csv(p)
    bad = [i for i in idx_set if i < 0 or i >= NUM_LABELS]
    mx = max(idx_set) if idx_set else None
    print(f"{split}.csv: max label index = {mx}, distinct indices = {len(idx_set)}, out_of_range = {bad or 'none'}")
    if bad:
        issues.append(f"{split}.csv has out-of-range indices: {bad}")

print()
if issues:
    print("Issues:")
    for x in issues:
        print(" -", x)
else:
    print("All num_labels / label_dict / YAML / artifact checks passed.")

label_dict.json entries : 28
src.utils NUM_LABELS    : 28 (EMOTION_LABELS len)
YAML num_labels         : 28
artifacts config.json   : num_labels = 28
train.csv: max label index = 27, distinct indices = 28, out_of_range = none
val.csv: max label index = 27, distinct indices = 28, out_of_range = none
test.csv: max label index = 27, distinct indices = 28, out_of_range = none

All num_labels / label_dict / YAML / artifact checks passed.
